In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import sys
sys.path.append('/workspace/user2/Final')

from model_base_14_final테스트용 import predict_emotion, eff_model, vit_model, DEVICE
# [주의] 모델 정의(EffNetArcFace, ViTArcFace 등)와 가중치 로드 코드가 
# 이 코드 위쪽 혹은 같은 파일 내에 반드시 포함되어 있어야 합니다.

모델 가중치를 로드합니다...
✅ 가중치 로드 완료.


In [2]:
# 앵커 데이터 로드
ANCHOR_FILE_PATH = "/workspace/user2/Final/감정조합임시_v3.csv"
IMG_PATH_INPUT = '/workspace/데이터셋/img/train/sadness/0h6a943225c61e4575d0cb3a9bb2de87914259f8c323c1a595992ad7d51e9lhw3.jpg' #임시

try:
    # '감정' 컬럼을 인덱스로 하여 로드
    df_anchors = pd.read_csv(ANCHOR_FILE_PATH, index_col='감정')
    print("✅ 감정 앵커 데이터를 성공적으로 로드했습니다.")
except Exception as e:
    print(f"❌ 앵커 데이터 로드 실패: {e}")
    df_anchors = None

def get_audience_score(model_result, target_emotion_name):
    """
    model_result: predict_emotion 함수의 리턴값 (Top-3 리스트)
    target_emotion_name: 비교할 기준 한글 감정 (예: '설렘')
    """
    if df_anchors is None:
        return "데이터프레임 로드 실패로 계산할 수 없습니다."
    
    if target_emotion_name not in df_anchors.index:
        return f"'{target_emotion_name}'은(는) 앵커 데이터에 존재하지 않습니다."

    # 1. 모델 결과를 5차원 벡터로 변환
    # 모델은 4종(A, H, P, S)만 추론하므로 Neutral은 0으로 설정
    emotion_map = {'Happy': 0.0, 'Sadness': 0.0, 'Anger': 0.0, 'Panic': 0.0, 'Neutral': 0.0}
    
    for item in model_result:
        emo_name = item['emotion']
        prob = float(item['probability']) / 100.0  # % -> 0~1 범위로 변환
        if emo_name in emotion_map:
            emotion_map[emo_name] = prob

    # [Happy, Sadness, Anger, Panic, Neutral] 순서로 벡터 생성
    audience_vector = np.array([
        emotion_map['Happy'], 
        emotion_map['Sadness'], 
        emotion_map['Anger'], 
        emotion_map['Panic'], 
        emotion_map['Neutral']
    ])

    # 2. 앵커 데이터에서 목표 감정 벡터 추출
    target_cols = ['Happy', 'Sadness', 'Anger', 'Panic', 'Neutral']
    anchor_vector = df_anchors.loc[target_emotion_name, target_cols].values.astype(float)

    # 3. 유클리드 거리 및 일치 점수 계산
    distance = np.linalg.norm(anchor_vector - audience_vector)
    max_dist = np.sqrt(2) # 합이 1인 두 벡터 사이의 최대 거리
    score = max(0, (1 - (distance / max_dist)) * 100)
    
    return {
        "Target": target_emotion_name,
        "Score": round(score, 2),
        "Distance": round(distance, 4)
    }

# ================= 7. 최종 통합 테스트 실행 =================
if __name__ == "__main__":
    img_path = IMG_PATH_INPUT
    
    test_img = cv2.imread(img_path)
    if test_img is None:
        print(f"❌ 이미지를 찾을 수 없습니다: {img_path}")
    else:
        # 1) 모델 추론 수행
        test_img_rgb = cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB)
        result = predict_emotion(test_img_rgb, eff_model, vit_model, DEVICE)
        
        # 2) 모델 추론 결과 출력
        print("\n" + "="*40)
        print(f"       [1] 모델 추론 결과 (Top-3)")
        print("="*40)
        for rank, data in enumerate(result, 1):
            print(f"   {rank}위: {data['emotion']:8s} ({data['probability']}%)")
        print("="*40)

        # 3) 감정 일치도 분석 (사용자 입력)
        if df_anchors is not None:
            print("\n💡 비교할 목표 감정을 입력해 주세요.")
            target_name = input("예(설렘, 씁쓸, 기대, 경외 등): ").strip()
            
            # 점수 산출
            final_score = get_audience_score(result, target_name)
            
            if isinstance(final_score, dict):
                print("\n" + "="*40)
                print(f"       [2] 관객 감정 분석 리포트")
                print("="*40)
                print(f" 🎯 분석 기준 감정 : {final_score['Target']}")
                print(f" ⭐ 감정 일치 점수 : {final_score['Score']}점")
                print(f" 📏 앵커와의 거리  : {final_score['Distance']}")
                print("="*40)
                
                # 분석 팁 출력
                if final_score['Score'] >= 80:
                    print("결과: 관객이 의도한 감정을 매우 정확하게 느끼고 있습니다!")
                elif final_score['Score'] >= 50:
                    print("결과: 의도한 감정과 어느 정도 유사한 정서가 관찰됩니다.")
                else:
                    print("결과: 의도한 감정보다는 다른 복합적인 감정이 강하게 나타납니다.")
            else:
                print(f"\n⚠️ 알림: {final_score}")

✅ 감정 앵커 데이터를 성공적으로 로드했습니다.



       [1] 모델 추론 결과 (Top-3)
   1위: Sadness  (49.03125%)
   2위: Panic    (18.203125%)
   3위: Happy    (17.0625%)

💡 비교할 목표 감정을 입력해 주세요.

       [2] 관객 감정 분석 리포트
 🎯 분석 기준 감정 : 경외
 ⭐ 감정 일치 점수 : 51.41점
 📏 앵커와의 거리  : 0.6872
결과: 의도한 감정과 어느 정도 유사한 정서가 관찰됩니다.
